# GELECTRA emotion pipeline (Colab)

Runs the same full-corpus flow as [`03_emotion_pipeline.py`](03_emotion_pipeline.py): load `df_combined.csv`, clean text, infer with the fine-tuned **3×8 emotions** GELECTRA head, **truncate to the first 512 subword tokens** per row (`run_emotion_inference` in `emotion_utils.py`), write a results CSV.

**What you need on the Colab VM**

1. This entire **`3a_Sentiment_Analysis`** folder (with `emotion_utils.py`, `03_emotion_pipeline.py`, `02_download_model_weights.py`, and `models/final/.../config.json`). Upload a zip and unzip, or copy the folder to Drive and mount it.
2. **`df_combined.csv`** (or set `DATA_CSV` to its path).

Use **Runtime → Change runtime type → GPU** for faster inference.

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys
from pathlib import Path

# Root folder that contains emotion_utils.py, 03_emotion_pipeline.py, models/, …
PROJECT_ROOT = Path("/content/3a_Sentiment_Analysis")

if not (PROJECT_ROOT / "emotion_utils.py").is_file():
    # Fallback: notebook run from inside the project directory
    _here = Path.cwd()
    if (_here / "emotion_utils.py").is_file():
        PROJECT_ROOT = _here
    else:
        raise FileNotFoundError(
            f"Could not find emotion_utils.py under {PROJECT_ROOT} or {Path.cwd()}. "
            "Unzip 3a_Sentiment_Analysis here or set PROJECT_ROOT to the correct path."
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT.resolve())

In [ ]:
# Install dependencies (same as requirements.txt)
req = PROJECT_ROOT / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

In [ ]:
import torch

print("CUDA:", torch.cuda.is_available(), "| Device count:", torch.cuda.device_count())

In [ ]:
# Download pytorch_model.bin (~445 MB) if missing
subprocess.check_call([sys.executable, str(PROJECT_ROOT / "02_download_model_weights.py")])

## Data path

- Set `DATA_CSV` in the next cell (e.g. `/content/df_combined.csv` after uploading the CSV to Colab).
- Or set `RUN_UPLOAD = True` in the cell below to upload `df_combined.csv` from your laptop.

In [ ]:
DATA_CSV = Path("/content/df_combined.csv")
OUTPUT_CSV = Path("/content/emotion_full_results.csv")

# Default: classify the Title column (512-token truncation at inference — same as CLI)
TEXT_COLUMN = "Title"
MIN_WORDS = 3
BATCH_SIZE = 32

In [ ]:
# Optional: set True to upload df_combined.csv from your laptop (updates DATA_CSV)
RUN_UPLOAD = False

if RUN_UPLOAD:
    try:
        from google.colab import files

        uploaded = files.upload()
        for name in uploaded:
            dest = Path("/content") / name
            dest.write_bytes(uploaded[name])
            DATA_CSV = dest
            print("Using", DATA_CSV)
    except ImportError:
        print("Not in Colab — set DATA_CSV manually.")
else:
    print("Skipping upload; using DATA_CSV =", DATA_CSV)

In [ ]:
_spec = importlib.util.spec_from_file_location(
    "emotion_pipeline_colab",
    PROJECT_ROOT / "03_emotion_pipeline.py",
)
_pipe = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_pipe)

MODEL_DIR = PROJECT_ROOT / "models" / "final" / "german-nlp-group" / "electra-base-german-uncased"

result = _pipe.run_pipeline(
    data_path=DATA_CSV,
    model_dir=MODEL_DIR,
    output_path=OUTPUT_CSV,
    batch_size=BATCH_SIZE,
    text_column=TEXT_COLUMN,
    min_words=MIN_WORDS,
)
result.head()

In [ ]:
try:
    from google.colab import files

    files.download(str(OUTPUT_CSV))
except ImportError:
    print("Saved to:", OUTPUT_CSV)